In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import copy
import random
import os
import warnings

# Suppress annoying warnings
warnings.filterwarnings("ignore")
os.makedirs('./outputs', exist_ok=True)

# ==========================================
# 1. UTILS & CONFIG
# ==========================================
def set_deterministic(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def get_device(): return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_class_subset(dataset, classes):
    indices = [i for i, label in enumerate(dataset.targets) if label in classes]
    return Subset(dataset, indices)

def build_replay_buffer(dataset, classes, per_class=600):
    # 50 classes * 600 = 30,000 max capacity
    indices = []
    counts = {k:0 for k in classes}
    for i, label in enumerate(dataset.targets):
        if label in classes and counts[label] < per_class:
            indices.append(i)
            counts[label] += 1
    return Subset(dataset, indices)

# ==========================================
# 2. INCREASED CAPACITY KFN ARCHITECTURE
# ==========================================
class CosineLinear(nn.Module):
    def __init__(self, in_features, out_features, sigma=10.0):
        super(CosineLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.sigma = nn.Parameter(torch.Tensor([sigma]))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

    def forward(self, input):
        return self.sigma * F.linear(F.normalize(input, p=2, dim=1), F.normalize(self.weight, p=2, dim=1))

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False), nn.BatchNorm2d(out_c))
    def forward(self, x): return F.relu(self.bn1(self.conv1(x)) + self.shortcut(x))

class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        
        # CAPACITY INCREASE: 64 -> 128
        self.final_conv = nn.Conv2d(256, 128, 1)
        
    def _make_layer(self, in_c, out_c, blocks, stride=1):
        layers = []
        layers.append(ResidualBlock(in_c, out_c, stride))
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = self.final_conv(x)
        return x

class StructuralFusionModule(nn.Module):
    def __init__(self, in_channels, old_dim, new_dim, novelty_score):
        super().__init__()
        self.novelty_score = novelty_score
        self.has_expansion = new_dim > 0
        
        self.proj_reuse = nn.Sequential(
            nn.Conv2d(in_channels, old_dim, 1, bias=False),
            nn.BatchNorm2d(old_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        self.gate_reuse = nn.Parameter(torch.tensor([0.0])) 
        
        if self.has_expansion:
            self.proj_expand = nn.Sequential(
                nn.Conv2d(in_channels, new_dim, 1, bias=False),
                nn.BatchNorm2d(new_dim),
                nn.ReLU()
            )

    def forward(self, x, old_memory_detached):
        reuse_scale = max(0.1, 1.0 - self.novelty_score)
        delta_old = self.proj_reuse(x) * torch.sigmoid(self.gate_reuse) * reuse_scale
        
        feat_new = None
        if self.has_expansion:
            raw_new = self.proj_expand(x)
            feat_new = raw_new - raw_new.mean(dim=(2,3), keepdim=True)
            
        return delta_old, feat_new

class GlobalModel(nn.Module):
    def __init__(self, n_specs, ch, n_classes, old_dim=128, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.old_n_specs = 1 
        self.old_proj = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in range(self.old_n_specs)])
        self.old_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(ch, ch//4, 1), nn.ReLU(), 
            nn.Conv2d(ch//4, ch, 1), nn.Sigmoid()
        )
        self.old_bottleneck = nn.Sequential(nn.Conv2d(ch, old_dim, 1), nn.ReLU())
        
        self.old_dim = old_dim
        self.new_dim = new_dim
        if new_dim > 0:
            self.fusion = StructuralFusionModule(ch, old_dim, new_dim, novelty_score)
        else:
            self.fusion = None
            
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = CosineLinear(old_dim + new_dim, n_classes)

    def forward_old(self, feats):
        f = feats[0] 
        p = self.old_proj[0](f)
        w = self.old_gate(p)
        z = p * w
        return self.old_bottleneck(z)

    def forward(self, feats):
        old_memory = self.forward_old(feats)
        
        if self.fusion is not None:
            spec_new = feats[-1]
            delta_old, feat_new = self.fusion(spec_new, old_memory.detach())
            enhanced_old = old_memory + delta_old
            if feat_new is not None:
                final_features_new_path = torch.cat([enhanced_old, feat_new], dim=1)
            else:
                final_features_new_path = enhanced_old
        else:
            final_features_new_path = old_memory

        # SCALED TO CIFAR-100: 50 Old Classes, 50 New Classes
        W_old = self.classifier.weight[:50, :self.old_dim]
        W_new = self.classifier.weight[50:, :]
        
        flat_old = self.pool(old_memory).flatten(1)
        norm_in_old = F.normalize(flat_old, p=2, dim=1)
        norm_W_old = F.normalize(W_old, p=2, dim=1)
        logits_old = F.linear(norm_in_old, norm_W_old) * self.classifier.sigma
        
        flat_new = self.pool(final_features_new_path).flatten(1)
        norm_in_new = F.normalize(flat_new, p=2, dim=1)
        norm_W_new = F.normalize(W_new, p=2, dim=1)
        logits_new = F.linear(norm_in_new, norm_W_new) * self.classifier.sigma
        
        return torch.cat([logits_old, logits_new], dim=1), old_memory

class KFN(nn.Module):
    def __init__(self, n_classes=50, n_specs=1, old_dim=128, new_dim=0, novelty_score=0.0):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist() for _ in range(n_specs)])
        # ch is now 128 to match Specialist output
        self.global_model = GlobalModel(n_specs, 128, n_classes, old_dim, new_dim, novelty_score)
    def forward(self, x): 
        return self.global_model([s(x) for s in self.specialists])

# ==========================================
# 3. EXPANSION & TRAINING PIPELINE
# ==========================================

def expand_kfn(old_model, trained_spec, new_dim, novelty_score):
    # SCALED TO CIFAR-100: Total 100 classes, matching old_dim=128
    new_kfn = KFN(n_classes=100, n_specs=2, old_dim=128, new_dim=new_dim, novelty_score=novelty_score).to(get_device())
    
    new_kfn.specialists[0].load_state_dict(old_model.specialists[0].state_dict())
    new_kfn.specialists[1].load_state_dict(trained_spec.state_dict())
    
    old_g = old_model.global_model
    new_g = new_kfn.global_model
    
    new_g.old_proj.load_state_dict(old_g.old_proj.state_dict())
    new_g.old_gate.load_state_dict(old_g.old_gate.state_dict())
    new_g.old_bottleneck.load_state_dict(old_g.old_bottleneck.state_dict())
    
    with torch.no_grad():
        # SCALED TO CIFAR-100: 50 Old Classes, 50 New Classes
        new_g.classifier.weight[:50, :128] = old_g.classifier.weight
        new_g.classifier.sigma.data = old_g.classifier.sigma.data.clone()
        nn.init.kaiming_normal_(new_g.classifier.weight[50:, :], nonlinearity='relu')
        new_g.classifier.weight[:50, 128:].zero_()
        
    return new_kfn

def compute_novelty(model, specialist, loader, device):
    model.eval()
    specialist.eval()
    scores = []
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            if getattr(device, "type", "") == 'cuda':
                with torch.amp.autocast('cuda'):
                    f_new = F.normalize(specialist(x).flatten(1), p=2, dim=1)
                    spec_feat = model.specialists[0](x)
                    f_old_map = model.global_model.forward_old([spec_feat])
                    f_old = F.normalize(f_old_map.flatten(1), p=2, dim=1)
            else:
                f_new = F.normalize(specialist(x).flatten(1), p=2, dim=1)
                spec_feat = model.specialists[0](x)
                f_old_map = model.global_model.forward_old([spec_feat])
                f_old = F.normalize(f_old_map.flatten(1), p=2, dim=1)

            proj = torch.sum(f_new * f_old, dim=1, keepdim=True) * f_old
            residual = f_new - proj
            scores.append(torch.norm(residual, p=2, dim=1).mean().item())
            
    avg = np.mean(scores)
    # CAPACITY INCREASE: Expansion dim increased to 256
    new_dim = 256 if avg > 0.25 else 128 
    return new_dim, avg

def evaluate(model, loader, device):
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if getattr(device, "type", "") == 'cuda':
                with torch.amp.autocast('cuda'):
                    logits, _ = model(x)
            else:
                logits, _ = model(x)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return 100 * correct / total

def run_kfn_cifar100(seed, loaders, epochs_A, epochs_B, device):
    set_deterministic(seed)
    train_A, train_B, test_A, test_B, replay_loader = loaders
    
    scaler = torch.amp.GradScaler('cuda') if getattr(device, "type", "") == 'cuda' else None

    # --- Phase 1: Base Network ---
    model = KFN(n_classes=50, n_specs=1, old_dim=128).to(device)
    opt1 = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=epochs_A)
    
    for _ in range(epochs_A):
        model.train()
        for x, y in train_A:
            x, y = x.to(device), y.to(device)
            opt1.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    logits, _ = model(x)
                    loss = F.cross_entropy(logits, y)
                scaler.scale(loss).backward(); scaler.step(opt1); scaler.update()
            else:
                logits, _ = model(x)
                loss = F.cross_entropy(logits, y)
                loss.backward(); opt1.step()
        sched1.step()
                
    acc_A_init = evaluate(model, test_A, device)

    # --- Phase 2: Specialist ---
    spec = Specialist().to(device)
    # Updated to 128 input dim matching new Specialist capacity
    head = nn.Linear(128, 50).to(device) 
    opt2 = torch.optim.AdamW(list(spec.parameters()) + list(head.parameters()), lr=1e-3)
    sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs_A)
    
    for _ in range(epochs_A): 
        spec.train()
        for x, y in train_B:
            x, y = x.to(device), (y - 50).to(device)
            opt2.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    logits = head(F.adaptive_avg_pool2d(spec(x), 1).flatten(1))
                    loss = F.cross_entropy(logits, y)
                scaler.scale(loss).backward(); scaler.step(opt2); scaler.update()
            else:
                logits = head(F.adaptive_avg_pool2d(spec(x), 1).flatten(1))
                loss = F.cross_entropy(logits, y)
                loss.backward(); opt2.step()
        sched2.step()

    # --- Expansion ---
    new_dim, novelty = compute_novelty(model, spec, test_B, device)
    model = expand_kfn(model, spec, new_dim, novelty)

    # --- Phase 3: Integration ---
    for n, p in model.named_parameters(): p.requires_grad = False
    
    for p in model.global_model.fusion.parameters(): p.requires_grad = True
    model.global_model.classifier.weight.requires_grad = True
    for p in model.global_model.old_gate.parameters(): p.requires_grad = True
    for p in model.global_model.old_bottleneck.parameters(): p.requires_grad = True
        
    opt3 = torch.optim.AdamW([
        {'params': model.global_model.fusion.parameters(), 'lr': 2e-3}, 
        {'params': model.global_model.old_gate.parameters(), 'lr': 5e-5}, 
        {'params': model.global_model.old_bottleneck.parameters(), 'lr': 5e-5},
        {'params': model.global_model.classifier.parameters(), 'lr': 2e-3}
    ], weight_decay=1e-4)
    sched3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=epochs_B)

    replay_iter = iter(replay_loader)
    
    for ep in range(epochs_B):
        model.train()
        model.specialists[0].eval() 
        model.global_model.old_proj.eval()
        model.specialists[1].eval()
        
        for x_B, y_B in train_B:
            x_B, y_B = x_B.to(device), y_B.to(device)
            
            try: x_A, y_A = next(replay_iter)
            except StopIteration: 
                replay_iter = iter(replay_loader)
                x_A, y_A = next(replay_iter)
            x_A, y_A = x_A.to(device), y_A.to(device)
            
            opt3.zero_grad()
            
            if scaler:
                with torch.amp.autocast('cuda'):
                    logits_B, _ = model(x_B)
                    loss_B = F.cross_entropy(logits_B, y_B)
                    
                    logits_A, _ = model(x_A)
                    loss_A = F.cross_entropy(logits_A, y_A)
                    
                    loss = 2.5 * loss_B + 0.8 * loss_A
                    
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                scaler.step(opt3)
                scaler.update()
            else:
                logits_B, _ = model(x_B)
                loss_B = F.cross_entropy(logits_B, y_B)
                logits_A, _ = model(x_A)
                loss_A = F.cross_entropy(logits_A, y_A)
                
                loss = 2.5 * loss_B + 0.8 * loss_A
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                opt3.step()
        sched3.step()

    # --- Phase 4: Bias Correction ---
    for p in model.parameters(): p.requires_grad = False
    model.global_model.classifier.weight.requires_grad = True
    opt4 = torch.optim.AdamW(model.global_model.classifier.parameters(), lr=5e-5, weight_decay=1e-4)
    sched4 = torch.optim.lr_scheduler.CosineAnnealingLR(opt4, T_max=5)
    
    for ep in range(5):
        model.train()
        for x, y in replay_loader:
            x, y = x.to(device), y.to(device)
            opt4.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    logits, _ = model(x)
                    loss = F.cross_entropy(logits, y)
                scaler.scale(loss).backward(); scaler.step(opt4); scaler.update()
            else:
                logits, _ = model(x)
                loss = F.cross_entropy(logits, y)
                loss.backward(); opt4.step()
        sched4.step()

    acc_A_final = evaluate(model, test_A, device)
    acc_B_final = evaluate(model, test_B, device)

    return {
        "acc_A_init": acc_A_init,
        "acc_A_final": acc_A_final,
        "acc_B_final": acc_B_final,
        "retention": (acc_A_final / acc_A_init) * 100 if acc_A_init > 0 else 0,
        "forgetting": acc_A_init - acc_A_final,
        "bwt": acc_A_final - acc_A_init
    }

# ==========================================
# 4. RUN ALL SEEDS & REPORT
# ==========================================
def run_all():
    device = get_device()
    print(f"🚀 Environment Configured: {device}")
    
    stats = ((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762))
    
    # STRONGER AUGMENTATION APPLIED
    t_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(*stats)
    ])
    t_test = transforms.Compose([transforms.ToTensor(), transforms.Normalize(*stats)])
    
    root = '/kaggle/input/cifar-100' if os.path.exists('/kaggle/input/cifar-100') else './data'
    train_ds = torchvision.datasets.CIFAR100(root=root, train=True, download=True, transform=t_train)
    test_ds = torchvision.datasets.CIFAR100(root=root, train=False, download=True, transform=t_test)
    
    EPOCHS_A = 30
    EPOCHS_B = 80
    BS = 128  # Safe fallback for OOM 
    NW = 2
    
    # Dataloaders scaled for CIFAR-100 & Increased buffer (600 per class)
    loaders = (
        DataLoader(get_class_subset(train_ds, range(50)), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True),
        DataLoader(get_class_subset(train_ds, range(50, 100)), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True),
        DataLoader(get_class_subset(test_ds, range(50)), batch_size=BS, num_workers=NW, pin_memory=True),
        DataLoader(get_class_subset(test_ds, range(50, 100)), batch_size=BS, num_workers=NW, pin_memory=True),
        DataLoader(build_replay_buffer(train_ds, range(50), per_class=600), batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, persistent_workers=True)
    )
    
    seeds = [0, 1, 2, 3, 4]
    results = []
    
    print("\n[1] Running Multi-Seed KFN Experiments...")
    for s in seeds:
        print(f" -> Seed {s}...")
        r = run_kfn_cifar100(s, loaders, EPOCHS_A, EPOCHS_B, device)
        results.append(r)
        print(f"    ↳ Seed {s} Results: Task A Init={r['acc_A_init']:.2f} | Task A Final={r['acc_A_final']:.2f} | Task B Final={r['acc_B_final']:.2f} | Retention={r['retention']:.2f}%")
        
    print("\n" + "="*80)
    print("SECTION A: KFN Baseline (Mean ± Std over 5 Seeds)")
    print("="*80)
    
    for k, label in [("acc_A_init", "Task A Init"), ("acc_A_final", "Task A Final"), ("acc_B_final", "Task B Final")]:
        vals = [r[k] for r in results]
        print(f"{label:15}: {np.mean(vals):.2f} ± {np.std(vals):.2f}")

    ret_vals = [r['retention'] for r in results]
    forg_vals = [r['forgetting'] for r in results]
    bwt_vals = [r['bwt'] for r in results]

    print(f"Retention:      {np.mean(ret_vals):.2f}% ± {np.std(ret_vals):.2f}")
    print(f"Forgetting:     {np.mean(forg_vals):.2f} ± {np.std(forg_vals):.2f}")
    print(f"Backward Trans: {np.mean(bwt_vals):.2f} ± {np.std(bwt_vals):.2f}")
    print("="*80)

if __name__ == "__main__":
    run_all()

🚀 Environment Configured: cuda


100%|██████████| 169M/169M [00:12<00:00, 13.3MB/s] 



[1] Running Multi-Seed KFN Experiments...
 -> Seed 0...
    ↳ Seed 0 Results: Task A Init=65.28 | Task A Final=60.62 | Task B Final=45.32 | Retention=92.86%
 -> Seed 1...
    ↳ Seed 1 Results: Task A Init=66.06 | Task A Final=60.84 | Task B Final=46.42 | Retention=92.10%
 -> Seed 2...
    ↳ Seed 2 Results: Task A Init=65.20 | Task A Final=59.16 | Task B Final=47.84 | Retention=90.74%
 -> Seed 3...
    ↳ Seed 3 Results: Task A Init=65.24 | Task A Final=59.88 | Task B Final=45.98 | Retention=91.78%
 -> Seed 4...
    ↳ Seed 4 Results: Task A Init=65.84 | Task A Final=59.72 | Task B Final=47.76 | Retention=90.70%

SECTION A: KFN Baseline (Mean ± Std over 5 Seeds)
Task A Init    : 65.52 ± 0.36
Task A Final   : 60.04 ± 0.61
Task B Final   : 46.66 ± 0.99
Retention:      91.64% ± 0.83
Forgetting:     5.48 ± 0.54
Backward Trans: -5.48 ± 0.54
